In [0]:
import os
import json
import urllib.request
import urllib.parse
from datetime import date, timedelta
from pyspark.sql import functions as F

In [0]:
catalogo = "databricks_cata_managed"
volume_landing = "cambio_ptax_raw_files"

data_fim = date.today()
data_inicio =  date.today()

data_inicio_api = data_inicio.strftime("%m-%d-%Y")
data_fim_api = data_fim.strftime("%m-%d-%Y")

data_inicio_ref = data_inicio.strftime("%Y-%m-%d")
data_fim_ref = data_fim.strftime("%Y-%m-%d")

batch_id = f"{data_inicio_ref}_{data_fim_ref}".replace("-", "")

moedas = ["USD", "EUR", "GBP", "JPY", "CAD", "AUD"]

landing_dir = f"/Volumes/{catalogo}/landing/{volume_landing}/batch_{batch_id}"

dbutils.fs.mkdirs(landing_dir)

print(f"Período: {data_inicio_ref} até {data_fim_ref}")
print(f"Landing dir: {landing_dir}")

In [0]:
def buscar_odata_paginado(url_base: str):
    registros = []
    url = url_base
    pagina = 1

    while url:
        print(f"Buscando página {pagina}: {url}")

        with urllib.request.urlopen(url, timeout=120) as response:
            payload = json.loads(response.read().decode("utf-8"))

        valores = payload.get("value", [])
        registros.extend(valores)

        url = payload.get("@odata.nextLink")
        pagina += 1

    return registros

In [0]:
for moeda in moedas:
    endpoint = (
        "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
        "CotacaoMoedaPeriodo(moeda=@moeda,dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    )

    parametros = {
        "@moeda": f"'{moeda}'",
        "@dataInicial": f"'{data_inicio_api}'",
        "@dataFinalCotacao": f"'{data_fim_api}'",
        "$format": "json",
        "$top": "100"
    }

    url = endpoint + "?" + urllib.parse.urlencode(parametros, safe="'@$")

    registros = buscar_odata_paginado(url)

    payload_saida = {
        "moeda": moeda,
        "data_inicio": data_inicio_ref,
        "data_fim": data_fim_ref,
        "batch_id": batch_id,
        "qtd_registros": len(registros),
        "registros": registros
    }

    arquivo_saida = f"{landing_dir}/ptax_{moeda}_{data_inicio_ref}_{data_fim_ref}.json"

    dbutils.fs.put(
        arquivo_saida,
        json.dumps(payload_saida, ensure_ascii=False),
        overwrite=True
    )

    print(f"Moeda {moeda}: {len(registros)} registros salvos em {arquivo_saida}")